#Dataset Dados Abertos STJ - Íntegras de Acórdãos do Diário da Justiça (Acórdãos e Decisões)

Pre-processing dos datasets por ano (único ou lista - 2021 ... 2026), parametrizado por tipo de decisão (monocrática ou acórdão) e por tipo de processo.

##Importação de bibliotecas e declaração de variáveis globais

In [ ]:
ENVIRONMENT = 'COLAB' #'PUCPR'
TIPO_ARQUIVO = 'decisoes' #'acordaos'
ANO_CARGA = 2021
#'AREsp' #'REsp' #HC
CLASSE = 'HC'

#Não sendo informado o ANO
ANO_INI =2021
ANO_FIM= 2026

import torch

# Check if a GPU (CUDA) is available
if torch.cuda.is_available():
    print("GPU is available!")
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not available, using CPU.")

# This is the line you'd use in your script to send your model or data to the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

GPU not available, using CPU.
Using device: cpu


##Parametrização conforme ambiente de execução.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%run "/content/drive/MyDrive/_Mestrado PUCPR/libs/config_environment.ipynb"

%run "/content/drive/MyDrive/_Mestrado PUCPR/libs/preprocessing_pucpr.ipynb"

Mounted at /content/drive
version 1.05.0


In [ ]:
from datetime import datetime, date

def cast_string_to_date(ano: str, date_string: str) -> date:
    """
    Casts a string in 'YYYY-MM-DD' format to a datetime.date object.

    Args:
        date_string (str): The date string to cast, e.g., '2023-10-26'.

    Returns:
        datetime.date: The converted date object.

    Raises:
        ValueError: If the date_string is not in 'YYYY-MM-DD' format or is not a valid date.
    """
    try:
        if (ano == 2022):
          return datetime.strptime(date_string[0:10], '%d/%m/%Y').date()
        if (ano == 2023):
          return datetime.strptime(date_string, '%d/%m/%Y %H:%M:%S').date()
        else:
          return datetime.strptime(date_string, '%Y-%m-%d').date()
    except ValueError:
        raise ValueError(f"Invalid date string format or value: '{date_string}'. Expected 'YYYY-MM-DD'.")
        #return None

# Example usage:
# try:
#     date_obj = cast_string_to_date('2023-10-26')
#     print(f"Converted date: {date_obj} (Type: {type(date_obj)})")
#     date_obj_invalid = cast_string_to_date('2023/10/26') # This will raise a ValueError
# except ValueError as e:
#     print(e)

##Carregar classe/rótulo para a decisão (Favorável ou Não ao Réu)

In [ ]:
# Define file paths
file_path = base_path + 'descricao-decisao-julgamento.csv'

# Load each CSV file into a dataframe
try:
    dfTipoDecisao = pd.read_csv(file_path, sep=';')
    print(f"Successfully loaded {file_path} into dataframe 'dfTipoDecisao'")
    display(dfTipoDecisao.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred while loading {file_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado PUCPR/Datasets/descricao-decisao-julgamento.csv into dataframe 'dfTipoDecisao'


,COD_TEOR_DECISAO,DESC_TEOR_DECISAO,DESC_RSULT_JGTO,RESULTADO_RESUMIDO,DECISAO_FAVORAVEL
0,1,Concedendo,"AGRAVO CONHECIDO EM PARTE E, NESSA PARTE, PROVIDO",PARCIALMENTE CONHECIDO E PROVIDO,S
1,1,Concedendo,Agravo regimental provido,PROVIDO,S
2,1,Concedendo,CONCEDENDO HABEAS CORPUS,CONCEDIDO,S
3,1,Concedendo,CONCEDENDO HABEAS DATA,CONCEDIDO,S
4,1,Concedendo,CONCEDENDO MANDADO DE SEGURANÇA,CONCEDIDA,S


##Carregar metadados dos acórdão ou decisões (Documento, Processo, Nro de Registro, Ministro, Teor).

In [ ]:
from datetime import datetime, date
import numpy as np
import pandas as pd

#tipo_arquivo: { acordaos = decisões colegiadas | decisoes = decisões monográticas }


def load_metadados(ano: int, tipo_arquivo: str = 'acordaos', classe: str = ''):

  def cast_string_to_date(date_string: str) -> date:
      """
      Casts a string in 'YYYY-MM-DD' format to a datetime.date object.

      Args:
          date_string (str): The date string to cast, e.g., '2023-10-26'.

      Returns:
          datetime.date: The converted date object.

      Raises:
          ValueError: If the date_string is not in 'YYYY-MM-DD' format or is not a valid date.
      """
      # Handle non-string values (like NaN floats) in date columns
      if not isinstance(date_string, str) and pd.isna(date_string):
          return pd.NaT # Return Not a Time for NaNs

      try:
          if (ano == 2022):
            return datetime.strptime(date_string[0:10], '%d/%m/%Y').date()
          if (ano == 2023):
            return datetime.strptime(date_string, '%d/%m/%Y %H:%M:%S').date()
          else:
            return datetime.strptime(date_string, '%Y-%m-%d').date()
      except ValueError:
          # If a ValueError still occurs for a string, return NaT as well
          return pd.NaT

  # Example usage:
  # try:
  #     date_obj = cast_string_to_date('2023-10-26')
  #     print(f"Converted date: {date_obj} (Type: {type(date_obj)})")
  #     date_obj_invalid = cast_string_to_date('2023/10/26') # This will raise a ValueError
  # except ValueError as e:
  #     print(e)

  def remove_blank(field):
    return field.strip() if isinstance(field, str) else field

  file_path = base_path + f'{ano}/metadados-{tipo_arquivo}-dj{ano}.csv'

  try:
      df_metadados = pd.read_csv(file_path, sep=';')
      print(f"Successfully loaded {file_path} into dataframe 'dfMetaDados-{tipo_arquivo}-' with {df_metadados.shape[0]} raws.")

      df_metadados['PROCESSO'].apply(remove_blank)
      df_metadados = df_metadados[df_metadados['PROCESSO'].astype(str).str.startswith(classe)]

      print(f"After clean for {classe} dataframe now has {df_metadados.shape[0]} raws.")

      #remover espaços em branco dos campos textuais
      df_metadados['TIPODOCUMENTO'].apply(remove_blank)
      df_metadados['NUMEROREGISTRO'].apply(remove_blank)

      df_metadados['RECURSO'].apply(remove_blank)
      df_metadados['TEOR'].apply(remove_blank)
      df_metadados['DESCRICAOMONOCRATICA'].apply(remove_blank)
      df_metadados['ASSUNTOS'].apply(remove_blank)

      df_metadados['DATAPUBLICACAO']   = df_metadados['DATAPUBLICACAO'].apply(cast_string_to_date)
      df_metadados['DATARECEBIMENTO']  = df_metadados['DATARECEBIMENTO'].apply(cast_string_to_date)
      df_metadados['DATADISTRIBUICAO'] = df_metadados['DATADISTRIBUICAO'].apply(cast_string_to_date)

      df_metadados['DATAPUBLICACAO'] = pd.to_datetime(df_metadados['DATAPUBLICACAO'])
      df_metadados['DATARECEBIMENTO'] = pd.to_datetime(df_metadados['DATARECEBIMENTO'])

      # Calculate the difference in days and convert to integer (allowing for NaNs)
      df_metadados['NUM_DIAS_CONCLUSAO_PROCESSO'] = (df_metadados['DATAPUBLICACAO'] - df_metadados['DATARECEBIMENTO']).dt.days.astype(pd.Int64Dtype())

      # Separar o número do processo pelo tipo
      df_metadados['TIPOPROCESSO'] = df_metadados['PROCESSO'].apply(lambda tipoprocesso: tipoprocesso.split(" ")[0])
      df_metadados['NUMEROPROCESSO'] = df_metadados['PROCESSO'].apply(lambda numeroprocesso: numeroprocesso.split(" ")[1])

      df_metadados.drop(columns=['PROCESSO'], inplace=True, axis=1)

      if (ano == 2024) and ('NM_MINISTRO' in df_metadados.columns) and (tipo_arquivo == 'acordaos'):
        print(" >> ajuste específico para 2024 e acordãos..")
        df_metadados.drop(columns=['MINISTRO'], inplace=True, axis=1)
        df_metadados.rename(columns={'NM_MINISTRO': 'MINISTRO'}, inplace=True)

      if (ano == 2021 or ano == 2025 or ano == 2026) and ('NM_MINISTRO' in df_metadados.columns):
        df_metadados.rename(columns={'NM_MINISTRO': 'MINISTRO'}, inplace=True)

      df_metadados['MINISTRO'].apply(remove_blank)

      #print("passou... ")
      #print(f"Total de linhas: {df_metadados.shape[0]}")

      return df_metadados
      #display(dfMetaDadosDecisao.head(3))
  except FileNotFoundError:
      print(f"Error: File not found at {file_path}")
      return pd.DataFrame() # Return an empty DataFrame on FileNotFoundError
  except Exception as e:
      print(f"An error occurred while loading {file_path}: {e}")
      return pd.DataFrame() # Return an empty DataFrame on any other Exception

In [ ]:
#Teste isoaldo para observar resultados
# ano = 2026
# df2 = load_metadados(ano, 'acordaos', 'HC')
# df2.head()
#dfMetaDadosAcordao.head()

#dfDetalhesAcordao = load_dados_acordaos(ano)
#dfDetalhesAcordao.head()

# df2.head(3)

In [ ]:
#Teste chamada direta
# ano = 2026
# file_path = base_path + f'{ano}/metadados-acordaos-dj{ano}.csv'
# print(file_path)

# df = pd.read_csv(file_path, sep=';')
# print(f"Successfully loaded {file_path} into dataframe 'dfMetaDadosDecisao' with {df.shape[0]} raws.")

In [ ]:
# df.head()

In [ ]:
#df[df['NM_MINISTRO'].notna()]

In [ ]:
#Teste chamada direta dados textuais
ano = 2022
#2/0
df_dados_teste = load_dados(ano, 'decisoes')
df_dados_teste.head()

/content/drive/MyDrive/_Mestrado PUCPR/Datasets/2022/dados-decisoes-dj2022.csv
Loading dataset (optimized for testing)...
Successfully loaded the first 50 rows of /content/drive/MyDrive/_Mestrado PUCPR/Datasets/2022/dados-decisoes-dj2022.csv


,SEQDOCUMENTO,ACORDAO,EMENTA,RELATORIO,VOTO,DECISAO
0,151735738,NaN,NaN,NaN,NaN,Cuida-se deagravo interposto por FREDERICO RAB...
1,150487061,NaN,NaN,NaN,NaN,Trata-se de agravo nos próprios autos interpos...
2,151515141,NaN,NaN,NaN,NaN,"Vistos, etc.Verifica-se que houve manifestação..."
3,151513289,NaN,NaN,NaN,NaN,"Vistos, etc.Verifica-se que houve manifestação..."
4,151280722,NaN,NaN,NaN,NaN,Cuida-se de agravo interno interposto por BENE...


In [ ]:
def extrair_relatorio(x):
    if not isinstance(x, str):
        return x

    x_upper = x.upper()
    # O marcador agora obrigatoriamente inclui o ponto final
    marker_rel = 'O RELATÓRIO.'

    if marker_rel in x_upper:
        rel_pos = x_upper.find(marker_rel)

        # Retorna todo o conteúdo antes da frase
        return x[:rel_pos].strip()
    return x

def extrair_decisao(x):
    if not isinstance(x, str):
        return x

    x_upper = x.upper()
    # O marcador agora obrigatoriamente inclui o ponto final
    marker_rel = 'O RELATÓRIO.'

    if marker_rel in x_upper:
        rel_pos = x_upper.find(marker_rel)
        start_idx = rel_pos + len(marker_rel)

        # Retorna todo o conteúdo que segue após o ponto-final da frase
        return x[start_idx:].strip()
    return x

In [20]:
# texto = """
# Cuida-se de habeas corpus com pedido de liminar impetrado em favor de ALEXSANDRO DOS SANTOS SANTANAem que se aponta como autoridade coatora o TRIBUNAL DE JUSTIÇA DO ESTADO DA BAHIA(HCn. 8036997-30.2020.8.05.0000).<br>Opaciente foi preso preventivamente, em 16/12/2020, e denunciado pela prática das condutas descritas nos art. 1, inciso I, "a", da Lei n. 9.455/97, c/c art. 129, § 9º, c/c art. 71, caput, art. 147, caput, c/c art. 71, caput, e art. 163, parágrafo único, I, c/c art. 69 do CP, com incidência da Lei n. 11.340/06.<br>Oimpetrante sustentaque o paciente está sofrendo constrangimento ilegal em razão daausência de fundamentação concreta do decreto prisional, ausência de audiência de custódia, evidente excesso de prazo para apreciação de pedido de revogação de prisão preventiva, bem comolesão ao princípio da presunção de inocência.<br>Requer, liminarmente, a revogação da prisão preventiva; no mérito, pugna pela concessão da ordem para que seja confirmada a liminar.<br>É, no essencial, o relatório.
# """

#2021
# texto = """
# DECISÃO<br>Cuida-se de habeas corpus com pedido de liminar impetrado em favor de ALEXSANDRO DOS SANTOS SANTANAem que se aponta como autoridade coatora o TRIBUNAL DE JUSTIÇA DO ESTADO DA BAHIA(HCn. 8036997-30.2020.8.05.0000).<br>Opaciente foi preso preventivamente, em 16/12/2020, e denunciado pela prática das condutas descritas nos art. 1, inciso I, "a", da Lei n. 9.455/97, c/c art. 129, § 9º, c/c art. 71, caput, art. 147, caput, c/c art. 71, caput, e art. 163, parágrafo único, I, c/c art. 69 do CP, com incidência da Lei n. 11.340/06.<br>Oimpetrante sustentaque o paciente está sofrendo constrangimento ilegal em razão daausência de fundamentação concreta do decreto prisional, ausência de audiência de custódia, evidente excesso de prazo para apreciação de pedido de revogação de prisão preventiva, bem comolesão ao princípio da presunção de inocência.<br>Requer, liminarmente, a revogação da prisão preventiva; no mérito, pugna pela concessão da ordem para que seja confirmada a liminar.<br>É, no essencial, o relatório. Decido.<br>A matéria não pode ser apreciada pelo Superior Tribunal de Justiça, pois não foi examinada pelo Tribunal de origem, que ainda não julgou o mérito do writ originário.<br>A jurisprudência do STJ firmou-se no sentido de que não cabe habeas corpus contra indeferimento de pedido de liminar em outro writ, salvo no caso de flagrante ilegalidade, conforme demonstra o seguinte precedente:<br>AGRAVO REGIMENTAL NO HABEAS CORPUS. EXECUÇÃO PENAL. CUMPRIMENTO DE PENA EM PRISÃO DOMICILIAR. RECOMENDAÇÃO 62/2020 DO CNJ. COVID-19.GRUPO DE RISCO. CRIME VIOLENTO. CONDIÇÃO DE SAÚDE. AUSÊNCIA DE DEMONSTRAÇÃO DE POSSIBILIDADE DE AGRAVAMENTO. RECÁLCULO DA PENA.INOVAÇÃO RECURSAL. SUPRESSÃO DE INSTÂNCIA. ILEGALIDADE. AUSÊNCIA. AGRAVO IMPROVIDO.<br> .. <br>3. A matéria relativa ao recálculo da pena para fins de progressão de regime, além de representar indevida inovação recursal, não foi objeto de análise pelo Tribunal de origem, motivo pelo qual esse ponto não poderá ser conhecido por esta Corte Superior, sob pena de indevida supressão de instância.<br>4. Agravo regimental improvido. (AgRg no HC n. 579.110/SP, relator Ministro Nefi Cordeiro, Sexta Turma, DJe de 14/9/2020.)<br>Confira-se também a Súmula n. 691 do STF: "Não compete ao Supremo Tribunal Federal conhecer de habeas corpus impetrado contra decisão do Relator que, em habeas corpus requerido a tribunal superior, indefere a liminar."<br>No caso, não visualizo, em juízo sumário, manifesta ilegalidade que autorize o afastamento da aplicação do mencionado verbete. Issoporque o próprio TJBA não visualizoupericulum in mora e fumus boni iuris para deferir a liminar, pois "verifica-se que os documentos acostados não permitem a este Julgador firmar um juízo de convicção que se dirija, sem sombra de dúvidas, ao quando suscitado pela Defesa, ao menos nesse primeiro contato com o presente writ. Assim, sem respaldo o pedido de provisão liminar, visto que os fundamentos que embasam o pedido de liminar tem natureza satisfativa, sendo impossível abranger a questão de mérito do writ sem ultrapassar os limites da cognição sumária, porque o caso demanda mais informações, a serem colhidas no momento oportuno, que não em sede de plantão judiciário" (e-STJ, fl. 297).<br>Ante o exposto, com fundamento no art. 21, XIII, c, c/c o art. 210 do RISTJ, indefiro liminarmente o presente habeas corpus.<br>Cientifique-se o Ministério Público Federal.<br>Publique-se. Intimem-se.
# """

#2022
texto = """
DECISÃO<br>Trata-se de agravo nos próprios autos interposto por METAL MAXI INDÚSTRIA E COMÉRCIO DE MOLAS E ARTEFATOS DE MADEIRA LTDA. EPP contra decisão que inadmitiu recurso especial contra acórdão assim ementado (e-STJ fls. 673/681):<br>Duplicatas mercantis. Demanda declaratória de inexigibilidade de títulos e reconvenção de cobrança. Improcedência da demanda principal e procedência parcial da reconvenção decretada em lº grau. Decisão alterada, à luz do conjunto probatório. Cerceamento de defesa não configurado. Comprovação de pagamento do débito representado por parte das duplicatas discriminadas na inicial. Não comprovadas as respectivas compras e vendas mercantis quanto aos demais títulos. Saques das duplicatas que se revelaram ilegítimos e os protestos indevidos. Demanda principal julgada procedente e reconvenção improcedente. Distribuição dos encargos de sucumbência alterada. Recurso provido, com determinação.<br>Os embargos de declaração foram rejeitados (e-STJ fls. 707/712).<br>Nas razões do especial (e-STJ fls. 745/767), interposto com fundamento no art. 105, III, "a", da CF, a recorrente aponta violação dos seguintes dispositivos legais:<br>(I) Arts. 17, 330 e 485, VI, do Código de Processo Civil, na medida em que "a ora Recorrente alegou sua ilegitimidade para figurar no polo passivo da presente demanda, haja vista que os títulos de crédito que dão lastro ao feito foram transferidos de sua titularidade para a empresa Kapital Factoring Sociedade de Fomento Comercial LTDA. por meio de contrato de factoring realizado entre as demandadas" (e-STJ 757).<br>(II) Art. 373 do CPC/2015, "tendo-se em conta que a premissa utilizada pelo Tribunal de Justiça de São Paulo, na definição do ônus probatório na causa, está equivocada" (e-STJ 762).<br>(III) Art. 20, § 3º, da Lei 5.474/68, uma vez que, "ao julgar que as duplicatas foram emitidas e protestadas sem negócios jurídicos causais, o acórdão enfrentado deturpou a presunção firmada em supramencionado dispositivo legal, de cuja exegese se conclui que o protesto somente é possível quando comprovado o negócio mercantil que deu causa ao título de crédito" (e-STJ 765). Decisão pela inadmissibilidade do recurso às fls. 815/816.<br>Razões do agravo às fls. 826/837. Afirma preenchidos os requisitos de admissibilidade do especial.<br>Contraminuta às fls. 841/848 (e-STJ).<br>É o relatório.<br>Decido.<br>No que se refere à alegada violação do art. 20, § 3º, da Lei 5.474/68, o conhecimento do recurso exige a reapreciação do conjunto fático-probatório, o que encontra óbice na Súmula n. 7 do STJ.<br>Com efeito, a tese jurídica suscitada no especial versa sobre temas relacionados a causalidade, protesto de duplicatas e a exeqüibilidade do título, matérias analisadas pelo Judiciário local à luz das provas contidas nos autos. Nesse sentido, calha ressaltar que para, sustentar suas razões, a recorrente indica fundamentos diretamente relacionados com os documentos juntados ao processo afirmando sua inaptidão para comprovar a relação jurídica subjacente ao título de crédito e, bem assim, para infirmar a presunção de veracidade decorrente do protesto.<br>Em relação à tese de violação dos arts. 17, 330, 373 e 485, VI, do CPC/2015, alega a recorrente que, após a transferência do crédito, por intermédio do contrato de "factoring", somente a faturizadora passa a ser parte legítima para responder por quaisquer demandas que envolvam discussões acerca do crédito representado pela cambial. Nesse sentido, na ação declaratória de inexistência de relação jurídica, somente poderia figurar como requerida a recorrida KAPITAL FACTORING SOCIEDADE DE FOMENTO COMERCIAL LTDA., que passou a ser a titular do crédito.<br>A jurisprudência do STJ, contudo, entende que, embora não tenha responsabilidade pela solvência do devedor do crédito cedido, o faturizado responde por sua existência e validade. Confira-se:<br>DIREITO CIVIL E EMPRESARIAL. CONTRATO DE FACTORING. CESSÃO DE CRÉDITO PRO SOLUTO. ARTS. 295 E 296 DO CÓDIGO CIVIL. GARANTIA DA EXISTÊNCIA DO CRÉDITO CEDIDO. DIREITO DE REGRESSO DA FACTORING RECONHECIDO.<br>1. Em regra, a empresa de factoring não tem direito de regresso contra a faturizada - com base no inadimplemento dos títulos transferidos -, haja vista que esse risco é da essência do contrato de factoring. Essa impossibilidade de regresso decorre do fato de que a faturizada não garante a solvência do título, o qual, muito pelo contrário, é garantido exatamente pela empresa de factoring.<br>2. Essa característica, todavia, não afasta a responsabilidade da cedente em relação à existência do crédito, pois tal garantia é própria da cessão de crédito comum - pro soluto. É por isso que a doutrina, de forma uníssona, afirma que no contrato de factoring e na cessão de crédito ordinária, a faturizada/cedente não garante a solvência do crédito, mas a sua existência sim. Nesse passo, o direito de regresso da factoring contra a faturizada deve ser reconhecido quando estiver em questão não um mero inadimplemento, mas a própria existência do crédito.<br>3. No caso, da moldura fática incontroversa nos autos, fica claro que as duplicatas que ensejaram o processo executivo são desprovidas de causa - "frias" -, e tal circunstância consubstancia vício de existência dos créditos cedidos - e não mero inadimplemento -, o que gera a responsabilidade regressiva da cedente perante a cessionária.<br>4. Recurso especial provido.<br>(REsp 1289995/PE, Rel. Ministro LUIS FELIPE SALOMÃO, QUARTA TURMA, julgado em 20/02/2014, DJe 10/06/2014)<br>Estando em controvérsia não a solvibilidade do devedor sacado, mas a própria validade do título, que carece de causa para sua emissão, o faturizado também é parte legítima para figurar no polo passivo da ação.<br>Nesses termos, o conhecimento do recurso depara-se com o óbice erigido pela nota n. 83 da Súmula do STJ.<br>Por fim, em relação à tese de ofensa ao art. 373, § 1º, do CPC/2015, o entendimento manifestado pelo TJ local - no sentido de que os recorrentes, demandados na ação declaratória de inexigibilidade, deveriam comprovar a existência da relação jurídica subjacente - encontra suporte na jurisprudência desta Corte Superior:<br>PROCESSUAL CIVIL. RECURSO ESPECIAL. PROTESTO DE DUPLICATA. EXISTÊNCIA DE RELAÇÃO JURÍDICA SUBJACENTE AO TÍTULO. ÔNUS DA PROVA. PROVA NEGATIVA. IMPOSSIBILIDADE MATERIAL. I. Tratando-se de alegação de inexistência de relação jurídica ensejadora da emissão do título protestado, impossível impor-se o ônus de prová-la ao autor, sob pena de determinar-se prova negativa, mesmo porque basta ao réu, que protestou referida cártula, no caso duplicata, demonstrar que sua emissão funda-se em efetiva entrega de mercadoria ou serviços, cuja prova é perfeitamente viável. Precedentes. II. Recurso especial conhecido e provido. (REsp 763.033/PR, Rel. Ministro Aldir Passarinho Junior, Quarta Turma, DJe 22/6/2010).<br>A irresignação, por conseguinte, igualmente esbarra na Súmula n. 83/STJ.<br>Ante o exposto, NEGO PROVIMENTO ao agravo nos próprios autos.<br>Na forma prevista pelo art. 85, § 11, do CPC/2015, MAJORO em 20% (vinte por cento) os honorários advocatícios arbitrados pelas instâncias ordinárias.<br>Publique-se. Intimem-se.
"""

decisao = extrair_decisao(texto)
print("decisão:" + decisao)

relatorio_content = extrair_relatorio(texto)
print("relatorio:" + relatorio_content)


decisão:<br>Decido.<br>No que se refere à alegada violação do art. 20, § 3º, da Lei 5.474/68, o conhecimento do recurso exige a reapreciação do conjunto fático-probatório, o que encontra óbice na Súmula n. 7 do STJ.<br>Com efeito, a tese jurídica suscitada no especial versa sobre temas relacionados a causalidade, protesto de duplicatas e a exeqüibilidade do título, matérias analisadas pelo Judiciário local à luz das provas contidas nos autos. Nesse sentido, calha ressaltar que para, sustentar suas razões, a recorrente indica fundamentos diretamente relacionados com os documentos juntados ao processo afirmando sua inaptidão para comprovar a relação jurídica subjacente ao título de crédito e, bem assim, para infirmar a presunção de veracidade decorrente do protesto.<br>Em relação à tese de violação dos arts. 17, 330, 373 e 485, VI, do CPC/2015, alega a recorrente que, após a transferência do crédito, por intermédio do contrato de "factoring", somente a faturizadora passa a ser parte legí

##Carregar detalhes das decisões

In [ ]:
def load_dados(ano: int, tipo_arquivo: str = 'acordaos'):
  # Define file paths
  file_path = base_path + f'{ano}/dados-{tipo_arquivo}-dj{ano}.csv'
  print(file_path)
  try:
      print("Loading dataset (optimized for testing)...")
      # Using nrows=50 to load only the necessary rows from disk
      df_dados_acordaos = pd.read_csv(file_path, sep=';', nrows=50)
      print(f"Successfully loaded the first 50 rows of {file_path}")

      # if (tipo_arquivo == 'acordaos'):
      #   print("ementa pre processing...")
      #   df_dados_acordaos['EMENTA'] = df_dados_acordaos['EMENTA'].apply(pre_processamento)

      #   print("relatorio pre processing...")
      #   df_dados_acordaos['RELATORIO'] = df_dados_acordaos['RELATORIO'].apply(pre_processamento)
      # elif tipo_arquivo == 'decisoes':
      #   print("Extraindo RELATORIO e DECISAO...")
      #   df_dados_acordaos['RELATORIO'] = df_dados_acordaos['DECISAO'].apply(extrair_relatorio)
      #   df_dados_acordaos['DECISAO'] = df_dados_acordaos['DECISAO'].apply(extrair_decisao)

      #   print("DECISAO e RELATORIO pre processing...")
      #   df_dados_acordaos['RELATORIO'] = df_dados_acordaos['RELATORIO'].apply(pre_processamento)
      #   df_dados_acordaos['DECISAO'] = df_dados_acordaos['DECISAO'].apply(pre_processamento)
      # else:
      #   raise Exception("Tipo de arquivo não suportado")

      return df_dados_acordaos
  except FileNotFoundError:
      print(f"Error: File not found at {file_path}")
  except Exception as e:
      print(f"An error occurred while loading {file_path}: {e}")

In [ ]:
# df = load_dados(2021, TIPO_ARQUIVO)
# df

In [ ]:
def join_datasets(dfMetaDados, dfDetalhes, tipo_arquivo):

  def get_decisao(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['DECISAO'].iloc[0]
      else:
          return None

  def get_relatorio(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['RELATORIO'].iloc[0]
      else:
          return None

  def get_acordao(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['ACORDAO'].iloc[0]
      else:
          return None

  def get_ementa(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not row.empty:
          return row.head(1)['EMENTA'].iloc[0]
      else:
          return None

  def get_voto(seq_documento):
      field = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      if not field.empty:
          return field.head(1)['VOTO'].iloc[0]
      else:
          return None

  def get_tipo_decisao(teor_decisao):
      tipo_decisao_row = dfTipoDecisao[dfTipoDecisao['DESC_TEOR_DECISAO'] == teor_decisao].head(1)
      if not tipo_decisao_row.empty:
          return tipo_decisao_row.head(1)['DECISAO_FAVORAVEL'].iloc[0]
      else:
          return None

  dfMetaDados['DECISAO_FAVORAVEL']  = dfMetaDados['TEOR'].apply(get_tipo_decisao)

  if (tipo_arquivo == 'acordaos'):
    dfMetaDados['ACORDAO'] = dfMetaDados['SEQDOCUMENTO'].apply(get_acordao)
    dfMetaDados['EMENTA']  = dfMetaDados['SEQDOCUMENTO'].apply(get_ementa)
    dfMetaDados['RELATORIO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_relatorio)
    dfMetaDados['VOTO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_voto)
  elif (tipo_arquivo == 'decisoes'):
    dfMetaDados['DECISAO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_decisao)
    dfMetaDados['RELATORIO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_relatorio)
  else:
    raise Exception("Tipo de arquivo não suportado")

  return dfMetaDados

In [ ]:
import gc

def clear_object(obj):
  # Delete the individual dataframes to free up memory
  del obj

  # Explicitly call the garbage collector
  gc.collect()

  print("Memory for df_metadados and df_dados has been released.")

In [ ]:
# df_metadados.head()

In [ ]:
all_df_joins = []

inicio, fim = 0, 0
if ANO_CARGA:
  inicio, fim = ANO_CARGA, ANO_CARGA+1 #apenas um ano
else:
  inicio, fim = ANO_INI, ANO_FIM+1 #todo período

print (f"Realizando carga de {inicio} a {fim} ...")
for ano in range(inicio, fim):
  print(f'Processing {ano} year...')

  print(f'   >> Processing MetaData for {ano}, {TIPO_ARQUIVO}, {CLASSE}...')
  df_metadados = load_metadados(ano, TIPO_ARQUIVO, CLASSE)
  #dfMetaDadosAcordao.head()

  print('   >> Processing Text Data...')
  df_dados = load_dados(ano, TIPO_ARQUIVO)
  #df_dados.head()

  print('   >> Join metadata and text data...')
  dfJoin = join_datasets(df_metadados, df_dados, TIPO_ARQUIVO)
  all_df_joins.append(dfJoin)

  clear_object(df_metadados)
  clear_object(df_dados)
  clear_object(dfJoin)

dataset = pd.concat(all_df_joins, ignore_index=True)
clear_object(all_df_joins)

dataset.head()

In [ ]:
#df_metadados = load_metadados(2024, TIPO_ARQUIVO)

In [ ]:
#df_metadados.shape

####Verificando distribuição dos dados propriamente dita

In [ ]:
total   = dataset.shape[0]
label_s = dataset[dataset['DECISAO_FAVORAVEL'] == 'S'].shape[0]
label_n = dataset[dataset['DECISAO_FAVORAVEL'] == 'N'].shape[0]

print(f'TOTAL  : {total}')
print(f'LABEL S: {label_s} - % {round(label_s/total*100, 2)}')
print(f'LABEL N: {label_n} - % {round(label_n/total*100, 2)}')

In [ ]:
descricao_periodo = 'geral' if (ANO_CARGA == None) else ANO_CARGA
descricao_periodo

In [ ]:
CLASSE

In [ ]:
descricao_periodo = 'geral' if (ANO_CARGA is None) else str(ANO_CARGA)

output_csv_path = f"{base_path}preprocessed_{TIPO_ARQUIVO}_{CLASSE}_{descricao_periodo}_.csv"
print(output_csv_path)

# Save the DataFrame to a CSV file
dataset.to_csv(output_csv_path, index=False, sep=';')

print(f"Preprocessed dataset saved to: {output_csv_path}")
print("First 5 rows of the saved DataFrame:")
display(dataset.head())

In [ ]:
import pandas as pd

# Define the path to the preprocessed CSV file
input_csv_path = output_csv_path

# Load the CSV file into a DataFrame
try:
    df_loaded = pd.read_csv(input_csv_path, sep=';')
    print(f"Successfully loaded {input_csv_path} into dataframe 'df_loaded'")
    display(df_loaded.head())
except FileNotFoundError:
    print(f"Error: File not found at {input_csv_path}")
except Exception as e:
    print(f"An error occurred while loading {input_csv_path}: {e}")

In [ ]:
df_loaded.info()

In [ ]:
df_loaded.shape

In [ ]:
!python --version

In [ ]:
import pandas as pd
import numpy as np
import torch

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
#Exibir o conteúdo do RELATÓRIO a partir do campo NUMEROREGISTRO
#202002861050

In [ ]:
record_to_display = 202503493057 # @param {type:"integer"}

filtered_record = dataset[dataset['NUMEROREGISTRO'] == record_to_display]

if not filtered_record.empty:
    relatorio_content = filtered_record['RELATORIO'].iloc[0]
    print(f"RELATORIO content for NUMEROREGISTRO {record_to_display}:\n")
    print(relatorio_content)
else:
    print(f"No record found with NUMEROREGISTRO: {record_to_display}")

In [ ]:
dataset_with_relatorio = dataset[dataset['RELATORIO'].notna()]
display(dataset_with_relatorio.head())

### Resumo da Etapa de Preparação do Dataset

Este documento apresenta um script desenvolvido na linguagem Python (versão 3.12), cuja finalidade é concluir a elaboração do *dataset* central desta pesquisa de mestrado. A principal tarefa aqui executada consiste na consolidação e integração de um vasto volume de dados jurídicos (metadados e teores de decisões), os quais foram extraídos na fase anterior do projeto por meio de técnicas de *web scraping*. Através do cruzamento e unificação de bases distintas (sub-datasets), o código junta os dados brutos em um único arquivo, estruturando-os em um formato relacional consistente para formar o *dataset* principal da pesquisa. Para viabilizar esse processo de integração de dados, fez-se o uso intensivo da biblioteca Pandas (versão 2.2.2) na manipulação dos *dataframes*, com o apoio complementar do NumPy (versão 2.0.2) e PyTorch (versão 2.10.0).

In [ ]:
total_relatorio_filled = dataset['RELATORIO'].count()
total_relatorio_not_filled = dataset['RELATORIO'].isnull().sum()

print(f"Total records with 'RELATORIO' filled: {total_relatorio_filled}")
print(f"Total records with 'RELATORIO' not filled: {total_relatorio_not_filled}")